In [1]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import openpyxl

In [2]:
main_dir = Path("class_folder/USPC_Locarno_Conversion")

### Locarno Dictionary

In [3]:
loc_df = pd.read_excel("class_folder/loc_sheets.xlsx")
loc_df = loc_df[["Cl.", "EN - Goods LOC (15-2025)"]]

print(loc_df.columns)

Index(['Cl.', 'EN - Goods LOC (15-2025)'], dtype='str')


In [4]:
col = "EN - Goods LOC (15-2025)"

mask = loc_df[col].astype(str).str.contains("Subclass Heading", na=False)

loc_df.loc[mask, col] = (
    loc_df.loc[mask, col]
    .str.split("\n")
    .str[1]
)

loc_df = loc_df[mask].reset_index(drop=True)

In [5]:
loc_df.to_json("class_folder/locarno_classes.json", orient="records", indent=2)

### USPC Ranges

In [6]:
USPC_RANGES = {
    "D1":  (100, 199),   # Edible Products (starts at 100)
    "D2":  (500, 999),   # Apparel (Historical gap: 1-499 abolished)
    "D3":  (200, 329),   # Travel Goods/Personal Belongings
    "D4":  (100, 199),   # Brushware
    "D5":  (1, 99),      # Textile/Paper Yard Goods
    "D6":  (300, 719),   # Furnishings (Seats, Beds, etc.)
    "D7":  (300, 700),   # Food Prep/Serving
    "D8":  (14, 499),    # Tools and Hardware
    "D9":  (414, 783),   # Packages/Containers (Very dense)
    "D10": (1, 132),     # Measuring/Testing Instruments
    "D11": (1, 228),     # Jewelry/Ornaments
    "D12": (1, 608),     # Transportation (Cars, Aircraft)
    "D13": (101, 199),   # Energy Production/Distribution
    "D14": (125, 511),   # Recording/Comm/Electronics (Your target class)
    "D15": (10, 199),    # Machines (Industrial)
    "D16": (100, 342),   # Photography/Optical
    "D17": (1, 99),      # Musical Instruments
    "D18": (1, 59),      # Printing/Office Machinery
    "D19": (1, 204),     # Office Supplies/Artists Materials
    "D20": (1, 99),      # Sales/Advertising
    "D21": (300, 852),   # Games/Toys/Sports (Very scattered)
    "D22": (100, 199),   # Arms/Hunting/Fishing
    "D23": (200, 422),   # Heating/Cooling/Fluid (Includes your D23366)
    "D24": (100, 234),   # Medical/Lab Equipment
    "D25": (1, 199),     # Building Units/Construction
    "D26": (1, 156),     # Lighting
    "D27": (100, 196),   # Tobacco/Smokers Supplies
    "D28": (7, 99),      # Cosmetic/Toilet Articles
    "D29": (100, 130),   # Safety/Protection/Rescue
    "D30": (101, 199),   # Animal Husbandry
    "D32": (1, 74),      # Washing/Cleaning/Drying
    "D34": (1, 39),      # Material Handling (Carts, etc.)
    "D99": (1, 99)       # Miscellaneous (Caskets, Safes)
}

In [7]:
with open(main_dir / "USPC_RANGES.json", "w", encoding="utf-8") as f:
    json.dump(USPC_RANGES, f, ensure_ascii=False, indent=2)

### USPC Locarno Conversion

In [8]:
D01 = [
    {"U.S. Subclass": "100",        "Locarno Class - Subclass": "01 - 06", "Notes": ""},
    {"U.S. Subclass": "101",        "Locarno Class - Subclass": "01 - 01", "Notes": ""},
    {"U.S. Subclass": "116 - 120",  "Locarno Class - Subclass": "01 - 01", "Notes": ""},
    {"U.S. Subclass": "122 - 129",  "Locarno Class - Subclass": "01 - 01", "Notes": ""},
]

In [9]:
D02 = [
    {"U.S. Subclass": "500 - 508", "Locarno Class - Subclass": "02 - 05", "Notes": ""},
    {"U.S. Subclass": "600 - 601", "Locarno Class - Subclass": "02 - 05", "Notes": ""},
    {"U.S. Subclass": "602 - 604", "Locarno Class - Subclass": "02 - 01", "Notes": ""},
    {"U.S. Subclass": "605",       "Locarno Class - Subclass": "02 - 05", "Notes": ""},
    {"U.S. Subclass": "606 - 609", "Locarno Class - Subclass": "02 - 05", "Notes": ""},
    {"U.S. Subclass": "610",       "Locarno Class - Subclass": "02 - 06", "Notes": ""},
    {"U.S. Subclass": "611 - 613", "Locarno Class - Subclass": "02 - 99", "Notes": ""},
    {"U.S. Subclass": "614 - 623", "Locarno Class - Subclass": "02 - 06", "Notes": ""},
    {"U.S. Subclass": "624 - 640", "Locarno Class - Subclass": "02 - 07", "Notes": ""},
    {"U.S. Subclass": "641 - 643", "Locarno Class - Subclass": "07 - 99", "Notes": ""},
    {"U.S. Subclass": "700 - 710", "Locarno Class - Subclass": "02 - 01", "Notes": ""},
    {"U.S. Subclass": "711",       "Locarno Class - Subclass": "24 - 04", "Notes": ""},
    {"U.S. Subclass": "712 - 716", "Locarno Class - Subclass": "02 - 01", "Notes": ""},
    {"U.S. Subclass": "717",       "Locarno Class - Subclass": "02 - 02", "Notes": ""},
    {"U.S. Subclass": "718 - 723", "Locarno Class - Subclass": "02 - 01", "Notes": ""},
    {
        "U.S. Subclass": "724 - 853",
        "Locarno Class - Subclass": "02 - 02",
        "Notes": 'Classify "detachable trimming" for clothing as 05-04.'
    },
    {"U.S. Subclass": "854 - 855", "Locarno Class - Subclass": "02 - 02", "Notes": ""},
    {"U.S. Subclass": "856",       "Locarno Class - Subclass": "02 - 07", "Notes": ""},
    {"U.S. Subclass": "857",       "Locarno Class - Subclass": "02 - 02", "Notes": ""},
    {"U.S. Subclass": "858 - 859", "Locarno Class - Subclass": "02 - 02", "Notes": ""},
    {"U.S. Subclass": "860 - 864", "Locarno Class - Subclass": "02 - 02", "Notes": ""},
    {"U.S. Subclass": "865 - 893", "Locarno Class - Subclass": "02 - 03", "Notes": ""},
    {"U.S. Subclass": "894",       "Locarno Class - Subclass": "02 - 03", "Notes": ""},
    {"U.S. Subclass": "895",       "Locarno Class - Subclass": "02 - 03", "Notes": ""},
    {"U.S. Subclass": "896 - 975", "Locarno Class - Subclass": "02 - 04", "Notes": ""},
    {"U.S. Subclass": "976",       "Locarno Class - Subclass": "02 - 07", "Notes": ""},
    {"U.S. Subclass": "977",       "Locarno Class - Subclass": "02 - 04", "Notes": ""},
    {"U.S. Subclass": "978",       "Locarno Class - Subclass": "02 - 07", "Notes": ""},
    {"U.S. Subclass": "979",       "Locarno Class - Subclass": "07 - 99", "Notes": ""},
    {"U.S. Subclass": "980 - 994", "Locarno Class - Subclass": "02 - 04", "Notes": ""},
    {"U.S. Subclass": "999",       "Locarno Class - Subclass": "02 - 99", "Notes": ""},
]

In [10]:
D03 = [
    {"U.S. Subclass": "1 - 4",       "Locarno Class - Subclass": "03 - 04", "Notes": ""},
    {
        "U.S. Subclass": "5 - 7",
        "Locarno Class - Subclass": "03 - 03",
        "Notes": 'Classify "crutch armrest" as 24-05.'
    },
    {
        "U.S. Subclass": "10",
        "Locarno Class - Subclass": "03 - 03",
        "Notes": 'Classify "crutch armrest" as 24-05.'
    },
    {"U.S. Subclass": "8 - 9",       "Locarno Class - Subclass": "24 - 05", "Notes": ""},
    {"U.S. Subclass": "11",          "Locarno Class - Subclass": "03 - 01", "Notes": ""},
    {"U.S. Subclass": "12 - 16",     "Locarno Class - Subclass": "03 - 03", "Notes": ""},
    {"U.S. Subclass": "17",          "Locarno Class - Subclass": "24 - 05", "Notes": ""},
    {"U.S. Subclass": "18 - 21",     "Locarno Class - Subclass": "03 - 01", "Notes": ""},
    {"U.S. Subclass": "22 - 29",     "Locarno Class - Subclass": "02 - 07", "Notes": ""},
    {
        "U.S. Subclass": "200 - 201",
        "Locarno Class - Subclass": "03 - 01",
        "Notes": 'Classify "animal snack bag" as 03-01.'
    },
    {
        "U.S. Subclass": "202",
        "Locarno Class - Subclass": "09 - 02",
        "Notes": 'Classify "flask" as 09-01.'
    },
    {"U.S. Subclass": "203.1 - 203.8", "Locarno Class - Subclass": "03 - 01", "Notes": ""},
    {"U.S. Subclass": "204 - 212",   "Locarno Class - Subclass": "03 - 01", "Notes": ""},
    {"U.S. Subclass": "213 - 214",   "Locarno Class - Subclass": "03 - 05", "Notes": ""},
    {"U.S. Subclass": "215",          "Locarno Class - Subclass": "03 - 05", "Notes": ""},
    {
        "U.S. Subclass": "216 - 217.12",
        "Locarno Class - Subclass": "03 - 01",
        "Notes": 'Classify "card case" as 03-01.'
    },
    {"U.S. Subclass": "272",          "Locarno Class - Subclass": "09 - 03", "Notes": ""},
    {"U.S. Subclass": "273 - 303",    "Locarno Class - Subclass": "03 - 01", "Notes": ""},
    {
        "U.S. Subclass": "304 - 314",
        "Locarno Class - Subclass": "09 - 04",
        "Notes": 'Classify "shopping baskets" as 03-01.'
    },
    {"U.S. Subclass": "315 - 327",    "Locarno Class - Subclass": "03 - 01", "Notes": ""},
    {"U.S. Subclass": "328",          "Locarno Class - Subclass": "03 - 99", "Notes": ""},
    {"U.S. Subclass": "328.1",        "Locarno Class - Subclass": "03 - 01", "Notes": ""},
    {"U.S. Subclass": "329",          "Locarno Class - Subclass": "09 - 03", "Notes": ""},
]

In [11]:
D04 = [
    {"U.S. Subclass": "100",        "Locarno Class - Subclass": "04 - 02", "Notes": ""},
    {"U.S. Subclass": "100",        "Locarno Class - Subclass": "04 - 03", "Notes": ""},
    {
        "U.S. Subclass": "101",
        "Locarno Class - Subclass": "04 - 02",
        "Notes": 'Classify "brush for electric toothbrush" as 04-02. '
                 'Classify "electric toothbrush" as 28-03.'
    },
    {"U.S. Subclass": "102 - 113",  "Locarno Class - Subclass": "04 - 02", "Notes": ""},
    {"U.S. Subclass": "114 - 115",  "Locarno Class - Subclass": "04 - 01", "Notes": ""},
    {
        "U.S. Subclass": "116 - 118",
        "Locarno Class - Subclass": "04 - 99",
        "Notes": 'Classify "paint roller" as 08-05.'
    },
    {
        "U.S. Subclass": "119 - 123",
        "Locarno Class - Subclass": "04 - 01",
        "Notes": 'Classify "lint rollers" as 07-05. '
                 'Classify "brushes used in cooking" as 04-01.'
    },
    {"U.S. Subclass": "124 - 138",  "Locarno Class - Subclass": "04 - 99", "Notes": ""},
    {
        "U.S. Subclass": "199",
        "Locarno Class - Subclass": "04 - 02",
        "Notes": 'Classify "bristles for toothbrush" as 04-02. '
                 'Classify "attachments for affixing brushes to their handles" as 04-01.'
    },
]


In [12]:
D05 = [
    {"U.S. Subclass": "1 - 3",     "Locarno Class - Subclass": "05 - 05", "Notes": ""},
    {"U.S. Subclass": "4 - 6",     "Locarno Class - Subclass": "32 - 01", "Notes": ""},
    {"U.S. Subclass": "7 - 10",    "Locarno Class - Subclass": "05 - 04", "Notes": ""},
    {"U.S. Subclass": "11 - 13",   "Locarno Class - Subclass": "05 - 02", "Notes": ""},
    {"U.S. Subclass": "14 - 17",   "Locarno Class - Subclass": "05 - 03", "Notes": ""},
    {"U.S. Subclass": "18",        "Locarno Class - Subclass": "05 - 04", "Notes": ""},
    {"U.S. Subclass": "19",        "Locarno Class - Subclass": "05 - 02", "Notes": ""},
    {"U.S. Subclass": "20 - 22",   "Locarno Class - Subclass": "05 - 03", "Notes": ""},
    {"U.S. Subclass": "23 - 24",   "Locarno Class - Subclass": "05 - 02", "Notes": ""},
    {"U.S. Subclass": "25 - 36",   "Locarno Class - Subclass": "05 - 03", "Notes": ""},
    {"U.S. Subclass": "37",        "Locarno Class - Subclass": "06 - 04", "Notes": ""},
    {"U.S. Subclass": "38 - 47",   "Locarno Class - Subclass": "05 - 03", "Notes": ""},
    {"U.S. Subclass": "48",        "Locarno Class - Subclass": "05 - 02", "Notes": ""},
    {"U.S. Subclass": "49 - 51",   "Locarno Class - Subclass": "05 - 03", "Notes": ""},
    {"U.S. Subclass": "52",        "Locarno Class - Subclass": "06 - 11", "Notes": ""},
    {"U.S. Subclass": "53",        "Locarno Class - Subclass": "05 - 03", "Notes": ""},
    {"U.S. Subclass": "54",        "Locarno Class - Subclass": "05 - 05", "Notes": ""},
    {"U.S. Subclass": "55 - 60",   "Locarno Class - Subclass": "05 - 03", "Notes": ""},
    {"U.S. Subclass": "61 - 62",   "Locarno Class - Subclass": "05 - 06", "Notes": ""},
    {"U.S. Subclass": "63 - 66",   "Locarno Class - Subclass": "05 - 04", "Notes": ""},
    {"U.S. Subclass": "99",        "Locarno Class - Subclass": "05 - 99", "Notes": ""},
]

In [13]:
D06 = [
    {"U.S. Subclass": "300 - 314", "Locarno Class - Subclass": "06 - 07", "Notes": ""},
    {"U.S. Subclass": "315 - 328", "Locarno Class - Subclass": "06 - 08", "Notes": ""},
    {"U.S. Subclass": "329", "Locarno Class - Subclass": "06 - 03", "Notes": ""},
    {"U.S. Subclass": "330 - 332", "Locarno Class - Subclass": "06 - 06", "Notes": ""},
    {
        "U.S. Subclass": "333 - 381",
        "Locarno Class - Subclass": "06 - 01",
        "Notes": 'Classify "commode chairs" as 23-07. Classify "toilet seat adapters for babies" as 23-07.'
    },
    {"U.S. Subclass": "382 - 395", "Locarno Class - Subclass": "06 - 02", "Notes": ""},
    {"U.S. Subclass": "403 - 404", "Locarno Class - Subclass": "06 - 04", "Notes": ""},
    {"U.S. Subclass": "405", "Locarno Class - Subclass": "06 - 04", "Notes": ""},
    {"U.S. Subclass": "406.1 - 406.6", "Locarno Class - Subclass": "06 - 04", "Notes": ""},
    {"U.S. Subclass": "407", "Locarno Class - Subclass": "06 - 04", "Notes": ""},
    {"U.S. Subclass": "512 - 521", "Locarno Class - Subclass": "06 - 06", "Notes": ""},
    {"U.S. Subclass": "522 - 523", "Locarno Class - Subclass": "23 - 08", "Notes": ""},
    {"U.S. Subclass": "524 - 528", "Locarno Class - Subclass": "06 - 04", "Notes": ""},
    {"U.S. Subclass": "531", "Locarno Class - Subclass": "06 - 04", "Notes": ""},
    {"U.S. Subclass": "532 - 533", "Locarno Class - Subclass": "23 - 08", "Notes": ""},
    {"U.S. Subclass": "534", "Locarno Class - Subclass": "06 - 04", "Notes": ""},
    {"U.S. Subclass": "535", "Locarno Class - Subclass": "23 - 08", "Notes": ""},
    {"U.S. Subclass": "536 - 540", "Locarno Class - Subclass": "23 - 08", "Notes": ""},
    {"U.S. Subclass": "541", "Locarno Class - Subclass": "06 - 04", "Notes": ""},
    {"U.S. Subclass": "542 - 545", "Locarno Class - Subclass": "23 - 08", "Notes": ""},
    {"U.S. Subclass": "546 - 550", "Locarno Class - Subclass": "23 - 08", "Notes": ""},
    {"U.S. Subclass": "551", "Locarno Class - Subclass": "06 - 13", "Notes": ""},
    {"U.S. Subclass": "552", "Locarno Class - Subclass": "06 - 04", "Notes": ""},
    {"U.S. Subclass": "575 - 581", "Locarno Class - Subclass": "06 - 10", "Notes": ""},
    {"U.S. Subclass": "582 - 594", "Locarno Class - Subclass": "06 - 11", "Notes": ""},
    {"U.S. Subclass": "595 - 600", "Locarno Class - Subclass": "06 - 13", "Notes": ""},
    {"U.S. Subclass": "602 - 603", "Locarno Class - Subclass": "06 - 13", "Notes": ""},
    {"U.S. Subclass": "601", "Locarno Class - Subclass": "06 - 09", "Notes": ""},
    {"U.S. Subclass": "604 - 605", "Locarno Class - Subclass": "06 - 09", "Notes": ""},
    {"U.S. Subclass": "606", "Locarno Class - Subclass": "06 - 09", "Notes": ""},
    {"U.S. Subclass": "607", "Locarno Class - Subclass": "06 - 13", "Notes": ""},
    {"U.S. Subclass": "608 - 610", "Locarno Class - Subclass": "06 - 99", "Notes": ""},
    {"U.S. Subclass": "611", "Locarno Class - Subclass": "06 - 13", "Notes": ""},
    {"U.S. Subclass": "612 - 625", "Locarno Class - Subclass": "06 - 13", "Notes": ""},
    {"U.S. Subclass": "626", "Locarno Class - Subclass": "06 - 04", "Notes": ""},
    {"U.S. Subclass": "627 - 628", "Locarno Class - Subclass": "06 - 04", "Notes": ""},
    {"U.S. Subclass": "629 - 635", "Locarno Class - Subclass": "06 - 04", "Notes": ""},
    {"U.S. Subclass": "640", "Locarno Class - Subclass": "25 - 03", "Notes": ""},
    {"U.S. Subclass": "641 - 656.19", "Locarno Class - Subclass": "06 - 03", "Notes": ""},
    {"U.S. Subclass": "657 - 671.4", "Locarno Class - Subclass": "06 - 04", "Notes": ""},
    {"U.S. Subclass": "672 - 683.1", "Locarno Class - Subclass": "06 - 04", "Notes": ""},
    {"U.S. Subclass": "684", "Locarno Class - Subclass": "06 - 03", "Notes": ""},
    {"U.S. Subclass": "685 - 699.4", "Locarno Class - Subclass": "06 - 03", "Notes": ""},
    {"U.S. Subclass": "699.5", "Locarno Class - Subclass": "06 - 06", "Notes": ""},
    {"U.S. Subclass": "700 - 701", "Locarno Class - Subclass": "06 - 06", "Notes": ""},
    {"U.S. Subclass": "702 - 719.4", "Locarno Class - Subclass": "06 - 06", "Notes": ""},
]

In [14]:
D07 = [
    {"U.S. Subclass": "213", "Locarno Class - Subclass": "07 - 99", "Notes": ""},
    {
        "U.S. Subclass": "300",
        "Locarno Class - Subclass": "07 - 01",
        "Notes": 'Hand manipulated utensils → 07-04. Beer pump machines → 31-00. Electric teapot → 07-02.'
    },
    {"U.S. Subclass": "300.1", "Locarno Class - Subclass": "07 - 01", "Notes": ""},
    {"U.S. Subclass": "300.2", "Locarno Class - Subclass": "07 - 06", "Notes": ""},
    {"U.S. Subclass": "301 - 305", "Locarno Class - Subclass": "09 - 01", "Notes": ""},
    {
        "U.S. Subclass": "306 - 322",
        "Locarno Class - Subclass": "07 - 01",
        "Notes": 'Classify "coffee machine other than for household purpose" as 31-00.'
    },
    {"U.S. Subclass": "323 - 324", "Locarno Class - Subclass": "07 - 02", "Notes": ""},
    {"U.S. Subclass": "325", "Locarno Class - Subclass": "31 - 00", "Notes": ""},
    {"U.S. Subclass": "326", "Locarno Class - Subclass": "07 - 02", "Notes": ""},
    {"U.S. Subclass": "327", "Locarno Class - Subclass": "07 - 02", "Notes": ""},
    {
        "U.S. Subclass": "328 - 367",
        "Locarno Class - Subclass": "07 - 02",
        "Notes": 'Baking machine → 31-00. Baking paper → 05-06.'
    },
    {"U.S. Subclass": "368", "Locarno Class - Subclass": "07 - 04", "Notes": ""},
    {"U.S. Subclass": "369 - 386", "Locarno Class - Subclass": "31 - 00", "Notes": ""},
    {"U.S. Subclass": "387 - 389", "Locarno Class - Subclass": "07 - 06", "Notes": ""},
    {"U.S. Subclass": "390", "Locarno Class - Subclass": "07 - 02", "Notes": ""},
    {"U.S. Subclass": "391 - 392.1", "Locarno Class - Subclass": "07 - 02", "Notes": ""},
    {"U.S. Subclass": "393 - 396", "Locarno Class - Subclass": "07 - 03", "Notes": ""},
    {"U.S. Subclass": "396.1 - 396.6", "Locarno Class - Subclass": "07 - 99", "Notes": ""},
    {"U.S. Subclass": "397 - 401.1", "Locarno Class - Subclass": "07 - 06", "Notes": ""},
    {"U.S. Subclass": "412 - 415", "Locarno Class - Subclass": "07 - 04", "Notes": ""},
    {"U.S. Subclass": "416", "Locarno Class - Subclass": "07 - 99", "Notes": ""},
    {"U.S. Subclass": "417", "Locarno Class - Subclass": "07 - 99", "Notes": ""},
]


In [15]:
D08 = [
    {"U.S. Subclass": "1 - 4", "Locarno Class - Subclass": "08 - 01", "Notes": ""},
    {"U.S. Subclass": "5 - 9", "Locarno Class - Subclass": "08 - 03", "Notes": ""},
    {"U.S. Subclass": "10 - 11", "Locarno Class - Subclass": "08 - 01", "Notes": ""},
    {"U.S. Subclass": "12", "Locarno Class - Subclass": "08 - 03", "Notes": ""},
    {"U.S. Subclass": "13", "Locarno Class - Subclass": "08 - 01", "Notes": ""},
    {"U.S. Subclass": "14 - 14.1", "Locarno Class - Subclass": "08 - 05", "Notes": ""},
    {"U.S. Subclass": "15 - 17", "Locarno Class - Subclass": "08 - 05", "Notes": ""},
    {"U.S. Subclass": "18", "Locarno Class - Subclass": "07 - 99", "Notes": ""},
    {"U.S. Subclass": "19", "Locarno Class - Subclass": "08 - 05", "Notes": ""},
    {"U.S. Subclass": "20", "Locarno Class - Subclass": "08 - 03", "Notes": ""},
    {"U.S. Subclass": "21 - 32", "Locarno Class - Subclass": "08 - 05", "Notes": ""},
    {
        "U.S. Subclass": "33 - 43",
        "Locarno Class - Subclass": "07 - 06",
        "Notes": 'Classify "bottle opener" and "jar opener" as 07-06. Classify "tin and can opener" as 07-99.'
    },
    {
        "U.S. Subclass": "33 - 43",
        "Locarno Class - Subclass": "07 - 99",
        "Notes": 'Classify "bottle opener" and "jar opener" as 07-06. Classify "tin and can opener" as 07-99.'
    },
    {"U.S. Subclass": "44 - 46", "Locarno Class - Subclass": "08 - 05", "Notes": ""},
    {
        "U.S. Subclass": "47",
        "Locarno Class - Subclass": "08 - 01",
        "Notes": 'Classify "awl" as 08-01. Classify "chisel" as 08-03. '
                 'Classify "wedge" as 08-03. Classify "nail set" as 08-08.9.'
    },
    {
        "U.S. Subclass": "47",
        "Locarno Class - Subclass": "08 - 03",
        "Notes": 'Classify "awl" as 08-01. Classify "chisel" as 08-03. '
                 'Classify "wedge" as 08-03. Classify "nail set" as 08-08.9.'
    },
    {
        "U.S. Subclass": "47",
        "Locarno Class - Subclass": "08 - 08",
        "Notes": 'Classify "awl" as 08-01. Classify "chisel" as 08-03. '
                 'Classify "wedge" as 08-03. Classify "nail set" as 08-08.9.'
    },
    {
        "U.S. Subclass": "48 - 50",
        "Locarno Class - Subclass": "08 - 05",
        "Notes": 'Stapler exception: office use → 19-02; other use → 08-08.'
    },
    {"U.S. Subclass": "51 - 56", "Locarno Class - Subclass": "08 - 05", "Notes": ""},
    {"U.S. Subclass": "57", "Locarno Class - Subclass": "08 - 03", "Notes": 'Classify "poultry shears" as 07-04.'},
    {"U.S. Subclass": "58 - 59", "Locarno Class - Subclass": "08 - 05", "Notes": ""},
    {"U.S. Subclass": "60", "Locarno Class - Subclass": "08 - 03", "Notes": ""},
    {"U.S. Subclass": "61", "Locarno Class - Subclass": "08 - 05", "Notes": ""},
    {"U.S. Subclass": "62", "Locarno Class - Subclass": "08 - 05", "Notes": ""},
    {"U.S. Subclass": "63", "Locarno Class - Subclass": "08 - 05", "Notes": ""},
    {"U.S. Subclass": "64 - 66", "Locarno Class - Subclass": "08 - 03", "Notes": ""},
    {"U.S. Subclass": "67 - 70", "Locarno Class - Subclass": "08 - 05", "Notes": ""},
    {"U.S. Subclass": "70.1", "Locarno Class - Subclass": "16 - 05", "Notes": ""},
    {"U.S. Subclass": "71 - 74", "Locarno Class - Subclass": "08 - 05", "Notes": ""},
    {"U.S. Subclass": "75 - 81", "Locarno Class - Subclass": "08 - 02", "Notes": ""},
    {"U.S. Subclass": "82 - 87", "Locarno Class - Subclass": "08 - 04", "Notes": ""},
    {"U.S. Subclass": "88 - 94", "Locarno Class - Subclass": "08 - 05", "Notes": 'Classify "sandpaper" as 05-06.'},
    {"U.S. Subclass": "95 - 104", "Locarno Class - Subclass": "08 - 03", "Notes": ""},
    {"U.S. Subclass": "105 - 107", "Locarno Class - Subclass": "08 - 05", "Notes": ""},
    {"U.S. Subclass": "300 - 329", "Locarno Class - Subclass": "08 - 06", "Notes": ""},
    {"U.S. Subclass": "330 - 348", "Locarno Class - Subclass": "08 - 07", "Notes": ""},
    {"U.S. Subclass": "349", "Locarno Class - Subclass": "08 - 08", "Notes": ""},
    {"U.S. Subclass": "350 - 353", "Locarno Class - Subclass": "11 - 05", "Notes": ""},
    {"U.S. Subclass": "354 - 355", "Locarno Class - Subclass": "08 - 05", "Notes": ""},
    {"U.S. Subclass": "356", "Locarno Class - Subclass": "09 - 06", "Notes": ""},
    {"U.S. Subclass": "357 - 378", "Locarno Class - Subclass": "08 - 11", "Notes": ""},
    {"U.S. Subclass": "379 - 381", "Locarno Class - Subclass": "08 - 05", "Notes": ""},
    {"U.S. Subclass": "382 - 399", "Locarno Class - Subclass": "08 - 08", "Notes": ""},
    {"U.S. Subclass": "400", "Locarno Class - Subclass": "08 - 11", "Notes": ""},
    {"U.S. Subclass": "401", "Locarno Class - Subclass": "10 - 06", "Notes": ""},
    {"U.S. Subclass": "402 - 404", "Locarno Class - Subclass": "08 - 09", "Notes": ""},
    {"U.S. Subclass": "499", "Locarno Class - Subclass": "08 - 99", "Notes": ""},
]


In [16]:
D09 = [
    {"U.S. Subclass": "414 - 433", "Locarno Class - Subclass": "09 - 03", "Notes": 'Classify "packaging for CD" as 14-99.'},
    {"U.S. Subclass": "434 - 438", "Locarno Class - Subclass": "09 - 07", "Notes": ""},
    {"U.S. Subclass": "439 - 441", "Locarno Class - Subclass": "07 - 06", "Notes": ""},
    {
        "U.S. Subclass": "442",
        "Locarno Class - Subclass": "09 - 10",
        "Notes": 'Shopping handle for bag → 09-10. Clip for packaging → 09-07. Ice cream drip guard → 09-99.'
    },
    {"U.S. Subclass": "443 - 445", "Locarno Class - Subclass": "09 - 01", "Notes": ""},
    {"U.S. Subclass": "446", "Locarno Class - Subclass": "09 - 06", "Notes": ""},
    {"U.S. Subclass": "447 - 456", "Locarno Class - Subclass": "09 - 07", "Notes": ""},
    {"U.S. Subclass": "457", "Locarno Class - Subclass": "09 - 03", "Notes": ""},
    {"U.S. Subclass": "499", "Locarno Class - Subclass": "09 - 99", "Notes": ""},
    {"U.S. Subclass": "500 - 502", "Locarno Class - Subclass": "09 - 03", "Notes": ""},
    {"U.S. Subclass": "503 - 505", "Locarno Class - Subclass": "09 - 07", "Notes": ""},
    {"U.S. Subclass": "506 - 508", "Locarno Class - Subclass": "09 - 03", "Notes": ""},
    {"U.S. Subclass": "516", "Locarno Class - Subclass": "07 - 01", "Notes": ""},
    {"U.S. Subclass": "517 - 543", "Locarno Class - Subclass": "09 - 01", "Notes": ""},
    {"U.S. Subclass": "544 - 546", "Locarno Class - Subclass": "07 - 01", "Notes": ""},
    {
        "U.S. Subclass": "547 - 575",
        "Locarno Class - Subclass": "09 - 01",
        "Notes": 'Can → 09-03. Capsule/package containing washing product → 07-05.'
    },
    {
        "U.S. Subclass": "600 - 681",
        "Locarno Class - Subclass": "09 - 01",
        "Notes": 'Bread moulds → 31-00. Moulds for chocolate and confectionery → 31-00.'
    },
    {"U.S. Subclass": "682 - 694", "Locarno Class - Subclass": "09 - 01", "Notes": ""},
    {"U.S. Subclass": "695 - 701", "Locarno Class - Subclass": "09 - 05", "Notes": ""},
    {"U.S. Subclass": "702 - 714", "Locarno Class - Subclass": "09 - 05", "Notes": ""},
    {"U.S. Subclass": "715 - 722", "Locarno Class - Subclass": "09 - 03", "Notes": ""},
    {"U.S. Subclass": "723 - 736", "Locarno Class - Subclass": "09 - 01", "Notes": ""},
    {"U.S. Subclass": "737", "Locarno Class - Subclass": "09 - 03", "Notes": ""},
    {"U.S. Subclass": "738 - 747", "Locarno Class - Subclass": "09 - 01", "Notes": ""},
    {"U.S. Subclass": "748 - 762", "Locarno Class - Subclass": "09 - 03", "Notes": ""},
    {"U.S. Subclass": "763 - 783", "Locarno Class - Subclass": "09 - 03", "Notes": ""},
]


In [17]:
D10 = [
    {"U.S. Subclass": "1 - 29", "Locarno Class - Subclass": "10 - 01", "Notes": ""},
    {"U.S. Subclass": "30 - 39", "Locarno Class - Subclass": "10 - 02", "Notes": ""},
    {"U.S. Subclass": "40 - 45", "Locarno Class - Subclass": "10 - 03", "Notes": ""},
    {"U.S. Subclass": "46", "Locarno Class - Subclass": "10 - 07", "Notes": ""},
    {"U.S. Subclass": "46.1", "Locarno Class - Subclass": "21 - 01", "Notes": ""},
    {"U.S. Subclass": "46.2 - 47", "Locarno Class - Subclass": "10 - 04", "Notes": ""},
    {"U.S. Subclass": "48", "Locarno Class - Subclass": "10 - 05", "Notes": ""},
    {
        "U.S. Subclass": "49 - 74",
        "Locarno Class - Subclass": "10 - 04",
        "Notes": 'Humidity measuring instrument → 10-05. Template → 19-08. '
                 'Thermostat → 10-05. Drawing compass → 19-06. '
                 'Office/drawing ruler → 19-06. Slide rule → 19-99. '
                 'Distress signal rocket → 22-03. Ambulance → 12-08. '
                 'Body temperature measuring devices → 10-05.'
    },
    {"U.S. Subclass": "75 - 76", "Locarno Class - Subclass": "10 - 05", "Notes": ""},
    {"U.S. Subclass": "77 - 103", "Locarno Class - Subclass": "10 - 04", "Notes": ""},
    {
        "U.S. Subclass": "104.1 - 119.1",
        "Locarno Class - Subclass": "10 - 05",
        "Notes": 'Signal instrument → 10-06. Marker → 10-06. '
                 'Mooring booms → 25-03. Mooring buoys → 08-08.'
    },
    {"U.S. Subclass": "119.2 - 119.4", "Locarno Class - Subclass": "10 - 06", "Notes": ""},
    {"U.S. Subclass": "120", "Locarno Class - Subclass": "10 - 06", "Notes": ""},
    {"U.S. Subclass": "121", "Locarno Class - Subclass": "10 - 06", "Notes": ""},
    {"U.S. Subclass": "122 - 132", "Locarno Class - Subclass": "10 - 07", "Notes": ""},
]


In [18]:
D11 = [
    {"U.S. Subclass": "1 - 94",     "Locarno Class - Subclass": "11 - 01", "Notes": ""},
    {"U.S. Subclass": "95 - 116",   "Locarno Class - Subclass": "11 - 03", "Notes": ""},
    {"U.S. Subclass": "117",        "Locarno Class - Subclass": "11 - 04", "Notes": ""},
    {"U.S. Subclass": "118",        "Locarno Class - Subclass": "11 - 02", "Notes": ""},
    {"U.S. Subclass": "119 - 120",  "Locarno Class - Subclass": "11 - 04", "Notes": ""},
    {"U.S. Subclass": "121 - 129",  "Locarno Class - Subclass": "11 - 05", "Notes": ""},
    {"U.S. Subclass": "130",        "Locarno Class - Subclass": "11 - 04", "Notes": ""},
    {"U.S. Subclass": "130.1",      "Locarno Class - Subclass": "06 - 99", "Notes": ""},
    {"U.S. Subclass": "131",        "Locarno Class - Subclass": "11 - 02", "Notes": ""},
    {"U.S. Subclass": "131.1",      "Locarno Class - Subclass": "11 - 02", "Notes": ""},
    {"U.S. Subclass": "132 - 164",  "Locarno Class - Subclass": "11 - 02", "Notes": ""},
    {"U.S. Subclass": "165 - 183",  "Locarno Class - Subclass": "11 - 05", "Notes": ""},
    {"U.S. Subclass": "184",        "Locarno Class - Subclass": "11 - 99", "Notes": ""},
    {"U.S. Subclass": "200 - 241",  "Locarno Class - Subclass": "02 - 07", "Notes": ""},
]


In [19]:
D12 = [
    {"U.S. Subclass": "1 - 6",     "Locarno Class - Subclass": "12 - 07", "Notes": ""},
    {"U.S. Subclass": "7",         "Locarno Class - Subclass": "12 - 14", "Notes": ""},
    {"U.S. Subclass": "8 - 11",    "Locarno Class - Subclass": "12 - 07", "Notes": ""},
    {"U.S. Subclass": "12 - 13",   "Locarno Class - Subclass": "12 - 13", "Notes": ""},
    {"U.S. Subclass": "14 - 15",   "Locarno Class - Subclass": "12 - 08", "Notes": 'Classify "ice machine for skating rink" as 12-13.'},
    {"U.S. Subclass": "16",        "Locarno Class - Subclass": "12 - 02", "Notes": ""},
    {"U.S. Subclass": "16.1",      "Locarno Class - Subclass": "12 - 06", "Notes": ""},
    {"U.S. Subclass": "17 - 20",   "Locarno Class - Subclass": "12 - 01", "Notes": ""},
    {"U.S. Subclass": "36 - 41",   "Locarno Class - Subclass": "12 - 03", "Notes": ""},
    {"U.S. Subclass": "42 - 51",   "Locarno Class - Subclass": "12 - 17", "Notes": ""},
    {"U.S. Subclass": "52",        "Locarno Class - Subclass": "12 - 04", "Notes": ""},
    {"U.S. Subclass": "82 - 100",  "Locarno Class - Subclass": "12 - 08", "Notes": "Classify car parking lifts in 12-05."},
    {"U.S. Subclass": "101 - 106", "Locarno Class - Subclass": "12 - 10", "Notes": 'Classify wheel cover as 12-16.'},
    {"U.S. Subclass": "107 - 114", "Locarno Class - Subclass": "12 - 11", "Notes": 'Classify "trailer for bicycle" as 12-11.'},
    {"U.S. Subclass": "115",       "Locarno Class - Subclass": "08 - 10", "Notes": ""},
    {"U.S. Subclass": "116 - 127", "Locarno Class - Subclass": "12 - 11", "Notes": ""},
    {"U.S. Subclass": "128 - 133", "Locarno Class - Subclass": "24 - 05", "Notes": ""},
    {"U.S. Subclass": "159 - 172", "Locarno Class - Subclass": "12 - 16", "Notes": ""},
    {"U.S. Subclass": "173",       "Locarno Class - Subclass": "12 - 08", "Notes": ""},
    {"U.S. Subclass": "174 - 180", "Locarno Class - Subclass": "12 - 16", "Notes": ""},
    {"U.S. Subclass": "181 - 183", "Locarno Class - Subclass": "12 - 16", "Notes": ""},
    {"U.S. Subclass": "184 - 189", "Locarno Class - Subclass": "12 - 16", "Notes": ""},
    {"U.S. Subclass": "190",       "Locarno Class - Subclass": "12 - 08", "Notes": ""},
    {"U.S. Subclass": "191 - 194", "Locarno Class - Subclass": "12 - 16", "Notes": ""},
    {"U.S. Subclass": "195",       "Locarno Class - Subclass": "12 - 08", "Notes": ""},
    {"U.S. Subclass": "196 - 214", "Locarno Class - Subclass": "12 - 16", "Notes": ""},
    {"U.S. Subclass": "215",       "Locarno Class - Subclass": "12 - 06", "Notes": ""},
    {"U.S. Subclass": "216 - 223", "Locarno Class - Subclass": "12 - 16", "Notes": ""},
    {"U.S. Subclass": "300",       "Locarno Class - Subclass": "12 - 06", "Notes": ""},
    {"U.S. Subclass": "301 - 318", "Locarno Class - Subclass": "12 - 06", "Notes": ""},
    {"U.S. Subclass": "319 - 345", "Locarno Class - Subclass": "12 - 07", "Notes": 'Flying boards → 12-07. Parachute → 29-02. Vehicle wing → 12-16.'},
    {"U.S. Subclass": "400 - 406", "Locarno Class - Subclass": "12 - 16", "Notes": ""},
    {"U.S. Subclass": "407",       "Locarno Class - Subclass": "12 - 11", "Notes": ""},
    {"U.S. Subclass": "408 - 414", "Locarno Class - Subclass": "12 - 16", "Notes": ""},
    {"U.S. Subclass": "414.1",     "Locarno Class - Subclass": "12 - 16", "Notes": ""},
    {"U.S. Subclass": "415 - 420", "Locarno Class - Subclass": "12 - 16", "Notes": ""},
    {"U.S. Subclass": "421",       "Locarno Class - Subclass": "06 - 01", "Notes": ""},
    {"U.S. Subclass": "422 - 426.1","Locarno Class - Subclass": "12 - 16", "Notes": ""},
    {"U.S. Subclass": "500 - 608", "Locarno Class - Subclass": "12 - 15", "Notes": ""},
]


In [20]:
D13 = [
    {"U.S. Subclass": "100",       "Locarno Class - Subclass": "23 - 05", "Notes": ""},
    {"U.S. Subclass": "101",       "Locarno Class - Subclass": "13 - 02", "Notes": ""},
    {"U.S. Subclass": "102",       "Locarno Class - Subclass": "13 - 04", "Notes": ""},
    {"U.S. Subclass": "103 - 111", "Locarno Class - Subclass": "13 - 02", "Notes": ""},
    {"U.S. Subclass": "112 - 116", "Locarno Class - Subclass": "13 - 01", "Notes": ""},
    {"U.S. Subclass": "117 - 124", "Locarno Class - Subclass": "13 - 02", "Notes": ""},
    {"U.S. Subclass": "125 - 133", "Locarno Class - Subclass": "13 - 03", "Notes": ""},
    {"U.S. Subclass": "134 - 136", "Locarno Class - Subclass": "26 - 07", "Notes": ""},
    {"U.S. Subclass": "137 - 167", "Locarno Class - Subclass": "13 - 03", "Notes": ""},
    {"U.S. Subclass": "168",       "Locarno Class - Subclass": "14 - 03", "Notes": ""},
    {"U.S. Subclass": "169 - 182", "Locarno Class - Subclass": "13 - 03", "Notes": ""},
    {"U.S. Subclass": "183",       "Locarno Class - Subclass": "13 - 99", "Notes": 'Classify "magnet for magnetic board" as 11-02.'},
    {"U.S. Subclass": "184",       "Locarno Class - Subclass": "13 - 03", "Notes": ""},
    {"U.S. Subclass": "199",       "Locarno Class - Subclass": "13 - 99", "Notes": ""},
]


In [21]:
D14 = [
    {"U.S. Subclass": "125 - 134", "Locarno Class - Subclass": "14 - 02", "Notes": ""},
    {"U.S. Subclass": "135",       "Locarno Class - Subclass": "14 - 01", "Notes": ""},
    {"U.S. Subclass": "137 - 140.11","Locarno Class - Subclass": "14 - 03", "Notes": ""},
    {"U.S. Subclass": "141.1 - 141.3","Locarno Class - Subclass": "14 - 01", "Notes": ""},
    {"U.S. Subclass": "142 - 153", "Locarno Class - Subclass": "14 - 03", "Notes": ""},
    {"U.S. Subclass": "154",       "Locarno Class - Subclass": "14 - 01", "Notes": ""},
    {"U.S. Subclass": "155",       "Locarno Class - Subclass": "14 - 03", "Notes": ""},
    {"U.S. Subclass": "157",       "Locarno Class - Subclass": "14 - 03", "Notes": 'Brackets for radio sets for vehicles → 14-06.'},
    {"U.S. Subclass": "158",       "Locarno Class - Subclass": "14 - 99", "Notes": ""},
    {"U.S. Subclass": "159",       "Locarno Class - Subclass": "14 - 03", "Notes": ""},
    {"U.S. Subclass": "160 - 184", "Locarno Class - Subclass": "14 - 01", "Notes": ""},
    {"U.S. Subclass": "185 - 198", "Locarno Class - Subclass": "14 - 03", "Notes": ""},
    {"U.S. Subclass": "199 - 209", "Locarno Class - Subclass": "14 - 01", "Notes": ""},
    {"U.S. Subclass": "209.1",     "Locarno Class - Subclass": "14 - 06", "Notes": ""},
    {"U.S. Subclass": "210 - 216", "Locarno Class - Subclass": "14 - 01", "Notes": ""},
    {"U.S. Subclass": "217",       "Locarno Class - Subclass": "14 - 01", "Notes": ""},
    {"U.S. Subclass": "218",       "Locarno Class - Subclass": "14 - 03", "Notes": ""},
    {"U.S. Subclass": "219 - 229", "Locarno Class - Subclass": "14 - 01", "Notes": ""},
    {"U.S. Subclass": "230 - 250", "Locarno Class - Subclass": "14 - 03", "Notes": 'Antenna ornament → 11-05. Supports for television apparatus → 14-06.'},
    {"U.S. Subclass": "251",       "Locarno Class - Subclass": "14 - 06", "Notes": ""},
    {"U.S. Subclass": "252 - 256", "Locarno Class - Subclass": "14 - 03", "Notes": ""},
    {"U.S. Subclass": "257 - 260", "Locarno Class - Subclass": "14 - 01", "Notes": ""},
    {"U.S. Subclass": "260.1",     "Locarno Class - Subclass": "14 - 03", "Notes": ""},
    {"U.S. Subclass": "261",       "Locarno Class - Subclass": "14 - 01", "Notes": ""},
    {"U.S. Subclass": "262 - 264", "Locarno Class - Subclass": "14 - 01", "Notes": ""},
    {"U.S. Subclass": "265",       "Locarno Class - Subclass": "14 - 03", "Notes": ""},
    {"U.S. Subclass": "299",       "Locarno Class - Subclass": "14 - 99", "Notes": ""},
    {"U.S. Subclass": "300 - 446", "Locarno Class - Subclass": "14 - 02", "Notes": ""},
    {"U.S. Subclass": "447",       "Locarno Class - Subclass": "14 - 02", "Notes": ""},
    {"U.S. Subclass": "448 - 450", "Locarno Class - Subclass": "14 - 02", "Notes": ""},
    {"U.S. Subclass": "451 - 452", "Locarno Class - Subclass": "14 - 02", "Notes": ""},
    {"U.S. Subclass": "453 - 456", "Locarno Class - Subclass": "14 - 02", "Notes": ""},
    {"U.S. Subclass": "457",       "Locarno Class - Subclass": "14 - 06", "Notes": ""},
    {"U.S. Subclass": "458",       "Locarno Class - Subclass": "14 - 99", "Notes": ""},
    {"U.S. Subclass": "459",       "Locarno Class - Subclass": "14 - 99", "Notes": ""},
    {"U.S. Subclass": "460 - 461", "Locarno Class - Subclass": "06 - 06", "Notes": ""},
    {"U.S. Subclass": "462 - 471", "Locarno Class - Subclass": "14 - 03", "Notes": ""},
    {"U.S. Subclass": "472 - 473", "Locarno Class - Subclass": "14 - 03", "Notes": ""},
    {"U.S. Subclass": "474 - 484.1","Locarno Class - Subclass": "14 - 05", "Notes": 'CD cutter → 14-01.'},
    {"U.S. Subclass": "485 - 495", "Locarno Class - Subclass": "14 - 04", "Notes": ""},
    {"U.S. Subclass": "496",       "Locarno Class - Subclass": "14 - 03", "Notes": ""},
    {"U.S. Subclass": "497",       "Locarno Class - Subclass": "14 - 01", "Notes": 'Audio/video appliance for reproducing images → 14-01.'},
    {"U.S. Subclass": "498 - 511", "Locarno Class - Subclass": "14 - 03", "Notes": 'Optical fiber connectors → 14-99.'},
]


In [22]:
D15 = [
    {"U.S. Subclass": "1 - 6",     "Locarno Class - Subclass": "15 - 01", "Notes": ""},
    {"U.S. Subclass": "7 - 9.3",   "Locarno Class - Subclass": "15 - 02", "Notes": ""},
    {"U.S. Subclass": "10 - 13",   "Locarno Class - Subclass": "15 - 02",
     "Notes": 'Agricultural tool → 15-03. Construction tool → 15-04. '
              'Machines for felling trees → 15-03. Harvesters (forest) → 15-03. '
              'Boring machines → 15-09. Snowblowers → 15-03. Leaf blowers → 15-05.'},
    {"U.S. Subclass": "14 - 18",   "Locarno Class - Subclass": "15 - 03", "Notes": ""},
    {"U.S. Subclass": "19 - 22",   "Locarno Class - Subclass": "15 - 04", "Notes": ""},
    {"U.S. Subclass": "23 - 25",   "Locarno Class - Subclass": "12 - 09", "Notes": ""},
    {"U.S. Subclass": "26 - 33",   "Locarno Class - Subclass": "15 - 03", "Notes": ""},
    {"U.S. Subclass": "66 - 78",   "Locarno Class - Subclass": "15 - 06", "Notes": ""},
    {"U.S. Subclass": "79 - 91",   "Locarno Class - Subclass": "15 - 07", "Notes": 'Ice cube trays → 07-10.'},
    {"U.S. Subclass": "122 - 130", "Locarno Class - Subclass": "15 - 09", "Notes": ""},
    {"U.S. Subclass": "131",       "Locarno Class - Subclass": "08 - 01", "Notes": ""},
    {"U.S. Subclass": "132 - 144.2","Locarno Class - Subclass": "15 - 09", "Notes": ""},
    {"U.S. Subclass": "145 - 146", "Locarno Class - Subclass": "15 - 10", "Notes": ""},
    {"U.S. Subclass": "147",       "Locarno Class - Subclass": "15 - 04", "Notes": ""},
    {"U.S. Subclass": "148 - 149", "Locarno Class - Subclass": "15 - 99", "Notes": ""},
    {"U.S. Subclass": "150 - 152", "Locarno Class - Subclass": "09 - 09", "Notes": ""},
    {"U.S. Subclass": "199",       "Locarno Class - Subclass": "15 - 99", "Notes": ""},
]


In [23]:
D16 = [
    {"U.S. Subclass": "100 - 101", "Locarno Class - Subclass": "16 - 06", "Notes": ""},
    {"U.S. Subclass": "130",       "Locarno Class - Subclass": "16 - 99", "Notes": ""},
    {"U.S. Subclass": "131 - 137", "Locarno Class - Subclass": "16 - 06", "Notes": ""},
    {"U.S. Subclass": "200 - 206", "Locarno Class - Subclass": "16 - 01", "Notes": ""},
    {"U.S. Subclass": "207",       "Locarno Class - Subclass": "16 - 06", "Notes": ""},
    {"U.S. Subclass": "208 - 209", "Locarno Class - Subclass": "16 - 01", "Notes": ""},
    {"U.S. Subclass": "210 - 213", "Locarno Class - Subclass": "16 - 06", "Notes": ""},
    {"U.S. Subclass": "214 - 215", "Locarno Class - Subclass": "16 - 01", "Notes": ""},
    {"U.S. Subclass": "216 - 218", "Locarno Class - Subclass": "16 - 06", "Notes": ""},
    {"U.S. Subclass": "219 - 220", "Locarno Class - Subclass": "16 - 05", "Notes": ""},
    {"U.S. Subclass": "221 - 236", "Locarno Class - Subclass": "16 - 02", "Notes": ""},
    {"U.S. Subclass": "237 - 245", "Locarno Class - Subclass": "16 - 05", "Notes": ""},
    {"U.S. Subclass": "246 - 247", "Locarno Class - Subclass": "16 - 04", "Notes": ""},
    {"U.S. Subclass": "248",       "Locarno Class - Subclass": "16 - 03", "Notes": ""},
    {"U.S. Subclass": "249 - 250", "Locarno Class - Subclass": "16 - 04", "Notes": ""},
    {"U.S. Subclass": "300",       "Locarno Class - Subclass": "16 - 06", "Notes": ""},
    {"U.S. Subclass": "301",       "Locarno Class - Subclass": "28 - 03", "Notes": ""},
    {"U.S. Subclass": "302 - 342", "Locarno Class - Subclass": "16 - 06", "Notes": ""},
]


In [24]:
D17 = [
    {"U.S. Subclass": "1 - 9",   "Locarno Class - Subclass": "17 - 01", "Notes": ""},
    {"U.S. Subclass": "10 - 11", "Locarno Class - Subclass": "17 - 02", "Notes": ""},
    {"U.S. Subclass": "12",      "Locarno Class - Subclass": "17 - 02", "Notes": ""},
    {"U.S. Subclass": "13",      "Locarno Class - Subclass": "17 - 02", "Notes": ""},
    {"U.S. Subclass": "14 - 21", "Locarno Class - Subclass": "17 - 03", "Notes": ""},
    {"U.S. Subclass": "22 - 23", "Locarno Class - Subclass": "17 - 04", "Notes": ""},
    {"U.S. Subclass": "24",      "Locarno Class - Subclass": "17 - 05", "Notes": ""},
    {"U.S. Subclass": "99",      "Locarno Class - Subclass": "17 - 99", "Notes": ""},
]


In [25]:
D18 = [
    {"U.S. Subclass": "1 - 12.3", "Locarno Class - Subclass": "18 - 01",
     "Notes": 'Currency converter → 19-99. Electronic payment terminals → 14-02.'},
    {"U.S. Subclass": "14 - 18",  "Locarno Class - Subclass": "19 - 02", "Notes": ""},
    {"U.S. Subclass": "19",       "Locarno Class - Subclass": "18 - 99", "Notes": ""},
    {"U.S. Subclass": "20 - 21",  "Locarno Class - Subclass": "18 - 01", "Notes": ""},
    {"U.S. Subclass": "24 - 33",  "Locarno Class - Subclass": "18 - 03", "Notes": ""},
    {"U.S. Subclass": "34.1 - 34.9","Locarno Class - Subclass": "18 - 04", "Notes": ""},
    {"U.S. Subclass": "35 - 35.1","Locarno Class - Subclass": "18 - 99", "Notes": ""},
    {"U.S. Subclass": "36 - 45",  "Locarno Class - Subclass": "16 - 03", "Notes": ""},
    {"U.S. Subclass": "46 - 49",  "Locarno Class - Subclass": "18 - 99", "Notes": ""},
    {"U.S. Subclass": "50 - 59",  "Locarno Class - Subclass": "18 - 02", "Notes": 'Computer printer → 14-02.'},
    {"U.S. Subclass": "99",       "Locarno Class - Subclass": "18 - 99", "Notes": ""},
]


In [26]:
D19 = [
    {"U.S. Subclass": "1 - 8",    "Locarno Class - Subclass": "19 - 01",
     "Notes": 'Pre-printed card → 19-05. Lottery form → 19-08.'},
    {"U.S. Subclass": "9 - 12",   "Locarno Class - Subclass": "19 - 08", "Notes": ""},
    {"U.S. Subclass": "20 - 25",  "Locarno Class - Subclass": "19 - 03", "Notes": ""},
    {"U.S. Subclass": "26 - 33",  "Locarno Class - Subclass": "19 - 04", "Notes": ""},
    {"U.S. Subclass": "34",       "Locarno Class - Subclass": "19 - 99", "Notes": ""},
    {"U.S. Subclass": "34.1 - 34.5","Locarno Class - Subclass": "06 - 06", "Notes": ""},
    {"U.S. Subclass": "37 - 40",  "Locarno Class - Subclass": "19 - 06", "Notes": ""},
    {"U.S. Subclass": "59 - 64",  "Locarno Class - Subclass": "19 - 07", "Notes": ""},
    {"U.S. Subclass": "65 - 72",  "Locarno Class - Subclass": "19 - 99",
     "Notes": 'Brass fasteners for paper and binder clips → 19-02.'},
    {"U.S. Subclass": "73 - 74",  "Locarno Class - Subclass": "19 - 06", "Notes": ""},
    {"U.S. Subclass": "75",       "Locarno Class - Subclass": "19 - 07", "Notes": ""},
    {"U.S. Subclass": "76 - 80",  "Locarno Class - Subclass": "19 - 99", "Notes": ""},
    {"U.S. Subclass": "81 - 85",  "Locarno Class - Subclass": "19 - 06", "Notes": ""},
    {"U.S. Subclass": "86 - 92",  "Locarno Class - Subclass": "19 - 02", "Notes": ""},
    {"U.S. Subclass": "93 - 94",  "Locarno Class - Subclass": "19 - 06", "Notes": ""},
    {"U.S. Subclass": "95 - 98",  "Locarno Class - Subclass": "19 - 02", "Notes": ""},
    {"U.S. Subclass": "99 - 100", "Locarno Class - Subclass": "19 - 99", "Notes": ""},
    {"U.S. Subclass": "101 - 111","Locarno Class - Subclass": "19 - 06", "Notes": ""},
    {"U.S. Subclass": "112 - 204","Locarno Class - Subclass": "19 - 06", "Notes": ""},
]


In [27]:
D20 = [
    {"U.S. Subclass": "1 - 9",   "Locarno Class - Subclass": "20 - 01", "Notes": ""},
    {"U.S. Subclass": "10",      "Locarno Class - Subclass": "20 - 03", "Notes": ""},
    {"U.S. Subclass": "11 - 12", "Locarno Class - Subclass": "19 - 08", "Notes": ""},
    {"U.S. Subclass": "13 - 17", "Locarno Class - Subclass": "20 - 03", "Notes": ""},
    {"U.S. Subclass": "18",      "Locarno Class - Subclass": "20 - 02", "Notes": ""},
    {"U.S. Subclass": "19 - 21", "Locarno Class - Subclass": "20 - 02", "Notes": ""},
    {"U.S. Subclass": "22 - 44", "Locarno Class - Subclass": "20 - 03", "Notes": ""},
    {"U.S. Subclass": "99",      "Locarno Class - Subclass": "20 - 99", "Notes": ""},
]


In [28]:
D21 = [
    {"U.S. Subclass": "300",        "Locarno Class - Subclass": "21 - 01", "Notes": ""},
    {"U.S. Subclass": "301 - 368",  "Locarno Class - Subclass": "21 - 01", "Notes": ""},
    {"U.S. Subclass": "369 - 370",  "Locarno Class - Subclass": "21 - 03", "Notes": ""},
    {"U.S. Subclass": "371 - 422",  "Locarno Class - Subclass": "21 - 01", "Notes": ""},
    {"U.S. Subclass": "423",        "Locarno Class - Subclass": "12 - 11", "Notes": ""},
    {"U.S. Subclass": "424 - 436",  "Locarno Class - Subclass": "21 - 01", "Notes": ""},
    {"U.S. Subclass": "437",        "Locarno Class - Subclass": "22 - 02", "Notes": ""},
    {"U.S. Subclass": "438 - 442",  "Locarno Class - Subclass": "21 - 01", "Notes": ""},
    {"U.S. Subclass": "443 - 444",  "Locarno Class - Subclass": "21 - 02", "Notes": ""},
    {"U.S. Subclass": "445 - 659",  "Locarno Class - Subclass": "21 - 01", "Notes": ""},
    {"U.S. Subclass": "660 - 661",  "Locarno Class - Subclass": "21 - 03", "Notes": ""},
    {"U.S. Subclass": "662 - 725",  "Locarno Class - Subclass": "21 - 02",
     "Notes": 'Classify "physical therapy exerciser" as 24-01. Classify "self-balancing boards", i.e. "motorized hover boards" as 12-11.'},
    {"U.S. Subclass": "726",        "Locarno Class - Subclass": "21 - 01", "Notes": ""},
    {"U.S. Subclass": "727 - 767",  "Locarno Class - Subclass": "21 - 02", "Notes": ""},
    {"U.S. Subclass": "768",        "Locarno Class - Subclass": "21 - 02", "Notes": ""},
    {"U.S. Subclass": "769 - 781",  "Locarno Class - Subclass": "21 - 02", "Notes": ""},
    {"U.S. Subclass": "782 - 784",  "Locarno Class - Subclass": "21 - 01", "Notes": ""},
    {"U.S. Subclass": "785 - 799.1","Locarno Class - Subclass": "21 - 02", "Notes": ""},
    {"U.S. Subclass": "799.2",      "Locarno Class - Subclass": "21 - 02", "Notes": ""},
    {"U.S. Subclass": "800 - 802",  "Locarno Class - Subclass": "21 - 02", "Notes": ""},
    {"U.S. Subclass": "803",        "Locarno Class - Subclass": "21 - 03", "Notes": ""},
    {"U.S. Subclass": "804 - 805",  "Locarno Class - Subclass": "29 - 02", "Notes": ""},
    {"U.S. Subclass": "806 - 807",  "Locarno Class - Subclass": "21 - 02", "Notes": ""},
    {"U.S. Subclass": "808 - 810",  "Locarno Class - Subclass": "21 - 02", "Notes": ""},
    {"U.S. Subclass": "811 - 833",  "Locarno Class - Subclass": "21 - 03", "Notes": ""},
    {"U.S. Subclass": "834 - 840",  "Locarno Class - Subclass": "21 - 04", "Notes": ""},
]

In [29]:
D22 = [
    {"U.S. Subclass": "100 - 111",  "Locarno Class - Subclass": "22 - 01", "Notes": ""},
    {"U.S. Subclass": "112",        "Locarno Class - Subclass": "22 - 03", "Notes": ""},
    {"U.S. Subclass": "113 - 114",  "Locarno Class - Subclass": "22 - 04", "Notes": ""},
    {"U.S. Subclass": "115 - 116",  "Locarno Class - Subclass": "22 - 03", "Notes": ""},
    {"U.S. Subclass": "117 - 118",  "Locarno Class - Subclass": "22 - 02", "Notes": ""},
    {"U.S. Subclass": "119 - 124",  "Locarno Class - Subclass": "22 - 06", "Notes": ""},
    {"U.S. Subclass": "125 - 133",  "Locarno Class - Subclass": "22 - 05", "Notes": ""},
    {"U.S. Subclass": "134 - 150",  "Locarno Class - Subclass": "22 - 05", "Notes": ""},
    {"U.S. Subclass": "199",        "Locarno Class - Subclass": "22 - 99", "Notes": ""},
]

In [30]:
D23 = [
    {"U.S. Subclass": "200 - 202",  "Locarno Class - Subclass": "23 - 01",
     "Notes": 'Classify "lavabo" as 23-06. Classify "sauna ladle" as 08-05. Classify "outdoor fountains" as 25-03.'},
    {"U.S. Subclass": "203",        "Locarno Class - Subclass": "23 - 07", "Notes": ""},
    {"U.S. Subclass": "204 - 206",  "Locarno Class - Subclass": "23 - 01", "Notes": ""},
    {"U.S. Subclass": "207 - 208",  "Locarno Class - Subclass": "23 - 01", "Notes": ""},
    {"U.S. Subclass": "209 - 211",  "Locarno Class - Subclass": "23 - 01", "Notes": ""},
    {"U.S. Subclass": "211.1",      "Locarno Class - Subclass": "09 - 01", "Notes": ""},
    {"U.S. Subclass": "211.2",      "Locarno Class - Subclass": "23 - 01", "Notes": ""},
    {"U.S. Subclass": "212",        "Locarno Class - Subclass": "08 - 05", "Notes": ""},
    {"U.S. Subclass": "213 - 230",  "Locarno Class - Subclass": "23 - 01", "Notes": ""},
    {"U.S. Subclass": "231 - 232",  "Locarno Class - Subclass": "15 - 02", "Notes": ""},
    {"U.S. Subclass": "233 - 258",  "Locarno Class - Subclass": "23 - 01", "Notes": ""},
    {"U.S. Subclass": "259",        "Locarno Class - Subclass": "23 - 07", "Notes": ""},
    {"U.S. Subclass": "260 - 269",  "Locarno Class - Subclass": "23 - 01", "Notes": ""},
    {"U.S. Subclass": "270 - 275",  "Locarno Class - Subclass": "23 - 02", "Notes": ""},
    {"U.S. Subclass": "276 - 293.1","Locarno Class - Subclass": "23 - 06", "Notes": ""},
    {"U.S. Subclass": "295 - 303",  "Locarno Class - Subclass": "23 - 07", "Notes": ""},
    {"U.S. Subclass": "304 - 308",  "Locarno Class - Subclass": "23 - 06", "Notes": ""},
    {"U.S. Subclass": "309",        "Locarno Class - Subclass": "23 - 07", "Notes": ""},
    {"U.S. Subclass": "310",        "Locarno Class - Subclass": "23 - 06", "Notes": ""},
    {"U.S. Subclass": "311",        "Locarno Class - Subclass": "23 - 07", "Notes": ""},
    {"U.S. Subclass": "312 - 313",  "Locarno Class - Subclass": "23 - 06", "Notes": ""},
    {"U.S. Subclass": "314",        "Locarno Class - Subclass": "23 - 03", "Notes": ""},
    {"U.S. Subclass": "315",        "Locarno Class - Subclass": "23 - 06", "Notes": ""},
    {"U.S. Subclass": "316",        "Locarno Class - Subclass": "25 - 03",
     "Notes": 'Classify "swimming pools" as 25-03. Classify "aquariums" as 30-02.'},
    {"U.S. Subclass": "316",        "Locarno Class - Subclass": "30 - 02",
     "Notes": 'Classify "swimming pools" as 25-03. Classify "aquariums" as 30-02.'},
    {"U.S. Subclass": "317 - 323",  "Locarno Class - Subclass": "23 - 03", "Notes": ""},
    {"U.S. Subclass": "324 - 327",  "Locarno Class - Subclass": "12 - 16", "Notes": ""},
    {"U.S. Subclass": "328",        "Locarno Class - Subclass": "23 - 03", "Notes": ""},
    {"U.S. Subclass": "329 - 331",  "Locarno Class - Subclass": "23 - 03", "Notes": ""},
    {"U.S. Subclass": "332",        "Locarno Class - Subclass": "23 - 03", "Notes": ""},
    {"U.S. Subclass": "333",        "Locarno Class - Subclass": "23 - 04", "Notes": ""},
    {"U.S. Subclass": "334 - 335",  "Locarno Class - Subclass": "23 - 03", "Notes": ""},
    {"U.S. Subclass": "336 - 341",  "Locarno Class - Subclass": "23 - 03", "Notes": ""},
    {"U.S. Subclass": "342 - 350",  "Locarno Class - Subclass": "23 - 05", "Notes": ""},
    {"U.S. Subclass": "351 - 365",  "Locarno Class - Subclass": "23 - 04", "Notes": ""},
    {"U.S. Subclass": "366 - 369",  "Locarno Class - Subclass": "23 - 04", "Notes": ""},
    {"U.S. Subclass": "370 - 385",  "Locarno Class - Subclass": "23 - 04", "Notes": ""},
    {"U.S. Subclass": "384",        "Locarno Class - Subclass": "07 - 08", "Notes": ""},
    {"U.S. Subclass": "385",        "Locarno Class - Subclass": "23 - 04", "Notes": ""},
    {"U.S. Subclass": "386 - 395",  "Locarno Class - Subclass": "23 - 03", "Notes": ""},
    {"U.S. Subclass": "396",        "Locarno Class - Subclass": "07 - 08", "Notes": ""},
    {"U.S. Subclass": "397 - 402",  "Locarno Class - Subclass": "23 - 03", "Notes": ""},
    {"U.S. Subclass": "403",        "Locarno Class - Subclass": "07 - 08", "Notes": ""},
    {"U.S. Subclass": "404",        "Locarno Class - Subclass": "23 - 03", "Notes": ""},
    {"U.S. Subclass": "405 - 406",  "Locarno Class - Subclass": "07 - 08", "Notes": ""},
    {"U.S. Subclass": "407",        "Locarno Class - Subclass": "23 - 03", "Notes": ""},
    {"U.S. Subclass": "408",        "Locarno Class - Subclass": "07 - 08", "Notes": ""},
    {"U.S. Subclass": "409 - 422",  "Locarno Class - Subclass": "23 - 03", "Notes": ""},
    {"U.S. Subclass": "499",        "Locarno Class - Subclass": "23 - 99", "Notes": ""},
]

In [31]:
D24 = [
    {"U.S. Subclass": "100 - 106",  "Locarno Class - Subclass": "24 - 01", "Notes": ""},
    {"U.S. Subclass": "107 - 108",  "Locarno Class - Subclass": "24 - 01", "Notes": ""},
    {"U.S. Subclass": "109",        "Locarno Class - Subclass": "24 - 02", "Notes": ""},
    {"U.S. Subclass": "110 - 110.6","Locarno Class - Subclass": "29 - 02", "Notes": ""},
    {"U.S. Subclass": "111 - 120",  "Locarno Class - Subclass": "24 - 02", "Notes": ""},
    {"U.S. Subclass": "121 - 125",  "Locarno Class - Subclass": "24 - 04", "Notes": ""},
    {"U.S. Subclass": "126",        "Locarno Class - Subclass": "02 - 01", "Notes": ""},
    {"U.S. Subclass": "127 - 132",  "Locarno Class - Subclass": "24 - 02", "Notes": ""},
    {"U.S. Subclass": "133 - 150",  "Locarno Class - Subclass": "24 - 02",
     "Notes": 'Classify "ear-piercing apparatus" as 28-03. Classify "menstrual cups" as 24-04. Classify "ultrasonic probes for medical purposes" as 24-01. Classify "ultrasonic diagnostic apparatus" as 24-01. Classify "tattooing apparatus" as 28-03.'},
    {"U.S. Subclass": "151",        "Locarno Class - Subclass": "24 - 01", "Notes": ""},
    {"U.S. Subclass": "152 - 154",  "Locarno Class - Subclass": "24 - 02", "Notes": ""},
    {"U.S. Subclass": "155 - 157",  "Locarno Class - Subclass": "24 - 03", "Notes": ""},
    {"U.S. Subclass": "158 - 162",  "Locarno Class - Subclass": "24 - 01", "Notes": ""},
    {"U.S. Subclass": "163",        "Locarno Class - Subclass": "24 - 01", "Notes": ""},
    {"U.S. Subclass": "164 - 170",  "Locarno Class - Subclass": "24 - 02", "Notes": ""},
    {"U.S. Subclass": "172 - 173",  "Locarno Class - Subclass": "24 - 02", "Notes": ""},
    {"U.S. Subclass": "171",        "Locarno Class - Subclass": "24 - 04", "Notes": ""},
    {"U.S. Subclass": "174",        "Locarno Class - Subclass": "24 - 99", "Notes": ""},
    {"U.S. Subclass": "175",        "Locarno Class - Subclass": "24 - 99", "Notes": ""},
    {"U.S. Subclass": "176 - 192",  "Locarno Class - Subclass": "24 - 02",
     "Notes": 'Classify "light therapy device" as 24-01. Classify "magnetic therapy device" as 24-99. Classify "toe separators" as 28-03. Classify "nursing pads" as 24-04. Classify "teeth whitening devices" as 28-03.'},
    {"U.S. Subclass": "193 - 196",  "Locarno Class - Subclass": "24 - 04", "Notes": ""},
    {"U.S. Subclass": "197 - 199",  "Locarno Class - Subclass": "07 - 01", "Notes": ""},
    {"U.S. Subclass": "200 - 206",  "Locarno Class - Subclass": "24 - 01", "Notes": ""},
    {"U.S. Subclass": "207 - 208",  "Locarno Class - Subclass": "07 - 99", "Notes": ""},
    {"U.S. Subclass": "209 - 215",  "Locarno Class - Subclass": "24 - 01", "Notes": ""},
    {"U.S. Subclass": "216 - 230",  "Locarno Class - Subclass": "24 - 02", "Notes": ""},
    {"U.S. Subclass": "231",        "Locarno Class - Subclass": "24 - 99", "Notes": ""},
    {"U.S. Subclass": "232 - 234",  "Locarno Class - Subclass": "24 - 99", "Notes": ""},
]


In [32]:
D25 = [
    {"U.S. Subclass": "1 - 34",     "Locarno Class - Subclass": "25 - 03", "Notes": ""},
    {"U.S. Subclass": "35 - 45",    "Locarno Class - Subclass": "25 - 02", "Notes": ""},
    {"U.S. Subclass": "46",         "Locarno Class - Subclass": "08 - 99", "Notes": ""},
    {"U.S. Subclass": "47.1 - 61",  "Locarno Class - Subclass": "25 - 02",
     "Notes": 'Classify "elevator cage" as 12-05. Classify "screens for infection control" as 06-06.'},
    {"U.S. Subclass": "62 - 66",    "Locarno Class - Subclass": "25 - 04", "Notes": ""},
    {"U.S. Subclass": "67",         "Locarno Class - Subclass": "08 - 99", "Notes": ""},
    {"U.S. Subclass": "68",         "Locarno Class - Subclass": "25 - 04", "Notes": ""},
    {"U.S. Subclass": "69",         "Locarno Class - Subclass": "25 - 04", "Notes": ""},
    {"U.S. Subclass": "100 - 101",  "Locarno Class - Subclass": "08 - 99", "Notes": ""},
    {"U.S. Subclass": "102 - 125",  "Locarno Class - Subclass": "25 - 01", "Notes": ""},
    {"U.S. Subclass": "126 - 163",  "Locarno Class - Subclass": "25 - 01", "Notes": ""},
    {"U.S. Subclass": "164",        "Locarno Class - Subclass": "25 - 01", "Notes": ""},
    {"U.S. Subclass": "199",        "Locarno Class - Subclass": "25 - 99", "Notes": ""},
]


In [33]:
D26 = [
    {"U.S. Subclass": "1",          "Locarno Class - Subclass": "26 - 04", "Notes": 'Classify "candle light for grave" as 26-01.'},
    {"U.S. Subclass": "2 - 7",      "Locarno Class - Subclass": "26 - 04", "Notes": ""},
    {"U.S. Subclass": "8",          "Locarno Class - Subclass": "26 - 02", "Notes": ""},
    {"U.S. Subclass": "9 - 23",     "Locarno Class - Subclass": "26 - 01", "Notes": ""},
    {"U.S. Subclass": "24",         "Locarno Class - Subclass": "26 - 03", "Notes": ""},
    {"U.S. Subclass": "25",         "Locarno Class - Subclass": "26 - 04", "Notes": ""},
    {"U.S. Subclass": "26",         "Locarno Class - Subclass": "26 - 05", "Notes": ""},
    {"U.S. Subclass": "27",         "Locarno Class - Subclass": "26 - 04", "Notes": ""},
    {"U.S. Subclass": "28 - 36",    "Locarno Class - Subclass": "26 - 06", "Notes": ""},
    {"U.S. Subclass": "37 - 50",    "Locarno Class - Subclass": "26 - 02", "Notes": ""},
    {"U.S. Subclass": "51 - 66",    "Locarno Class - Subclass": "26 - 05", "Notes": ""},
    {"U.S. Subclass": "67",         "Locarno Class - Subclass": "25 - 03", "Notes": ""},
    {"U.S. Subclass": "68 - 74",    "Locarno Class - Subclass": "26 - 05", "Notes": ""},
    {"U.S. Subclass": "75 - 79",    "Locarno Class - Subclass": "26 - 04", "Notes": ""},
    {"U.S. Subclass": "80 - 84",    "Locarno Class - Subclass": "26 - 05", "Notes": ""},
    {"U.S. Subclass": "85 - 112",   "Locarno Class - Subclass": "26 - 07", "Notes": ""},
    {"U.S. Subclass": "113 - 117",  "Locarno Class - Subclass": "26 - 04", "Notes": ""},
    {"U.S. Subclass": "118 - 156",  "Locarno Class - Subclass": "26 - 07", "Notes": ""},
]


In [34]:
D27 = [
    {"U.S. Subclass": "100 - 101",  "Locarno Class - Subclass": "27 - 01", "Notes": 'Classify "electronic cigarettes" as 27-07.'},
    {"U.S. Subclass": "102 - 138",  "Locarno Class - Subclass": "27 - 03", "Notes": ""},
    {"U.S. Subclass": "139 - 161",  "Locarno Class - Subclass": "27 - 05", "Notes": ""},
    {"U.S. Subclass": "162",        "Locarno Class - Subclass": "27 - 07", "Notes": ""},
    {"U.S. Subclass": "163 - 171",  "Locarno Class - Subclass": "27 - 02", "Notes": ""},
    {"U.S. Subclass": "172 - 179",  "Locarno Class - Subclass": "27 - 06", "Notes": ""},
    {"U.S. Subclass": "180 - 182",  "Locarno Class - Subclass": "27 - 07", "Notes": ""},
    {"U.S. Subclass": "183 - 196",  "Locarno Class - Subclass": "27 - 99", "Notes": ""},
]


In [35]:
D28 = [
    {"U.S. Subclass": "4 - 8.2",    "Locarno Class - Subclass": "28 - 02",
     "Notes": 'Classify "block of washing product" as 28-99. Classify "lipstick cases" as 28-02.'},
    {"U.S. Subclass": "9",          "Locarno Class - Subclass": "28 - 03", "Notes": ""},
    {"U.S. Subclass": "9 - 43",     "Locarno Class - Subclass": "28 - 06", "Notes": ""},
    {"U.S. Subclass": "44 - 54",    "Locarno Class - Subclass": "28 - 03", "Notes": ""},
    {"U.S. Subclass": "54.1",       "Locarno Class - Subclass": "23 - 08", "Notes": ""},
    {"U.S. Subclass": "55",         "Locarno Class - Subclass": "24 - 02", "Notes": ""},
    {"U.S. Subclass": "56 - 73",    "Locarno Class - Subclass": "28 - 03", "Notes": ""},
    {"U.S. Subclass": "74 - 84",    "Locarno Class - Subclass": "28 - 06", "Notes": ""},
    {"U.S. Subclass": "85 - 91.1",  "Locarno Class - Subclass": "28 - 03", "Notes": ""},
    {"U.S. Subclass": "91.2",       "Locarno Class - Subclass": "28 - 99", "Notes": ""},
    {"U.S. Subclass": "92 - 93",    "Locarno Class - Subclass": "28 - 04", "Notes": ""},
    {"U.S. Subclass": "99",         "Locarno Class - Subclass": "28 - 99", "Notes": ""},
]


In [36]:
D29 = [
    {"U.S. Subclass": "100 - 101.5","Locarno Class - Subclass": "29 - 02",
     "Notes": 'Classify "protective type garment" as 02-02; 02-04; or 02-06. See the Locarno Handbook. Classify "body harness" as 30-04. Classify "emergency equipment posts" as 29-99.'},
    {"U.S. Subclass": "102 - 107",  "Locarno Class - Subclass": "29 - 02",
     "Notes": 'Classify "helmet" as 02-03. Classify "spoiler for helmet" as 02-03.'},
    {"U.S. Subclass": "108",        "Locarno Class - Subclass": "29 - 06", "Notes": ""},
    {"U.S. Subclass": "109 - 112",  "Locarno Class - Subclass": "29 - 02", "Notes": ""},
    {"U.S. Subclass": "113 - 119",  "Locarno Class - Subclass": "02 - 06", "Notes": 'Classify "massage glove or horsehair type glove" as 28-03.'},
    {"U.S. Subclass": "120.1",      "Locarno Class - Subclass": "28 - 02", "Notes": ""},
    {"U.S. Subclass": "120.2",      "Locarno Class - Subclass": "08 - 07", "Notes": ""},
    {"U.S. Subclass": "121.1",      "Locarno Class - Subclass": "02 - 99", "Notes": ""},
    {"U.S. Subclass": "121.2",      "Locarno Class - Subclass": "24 - 02", "Notes": ""},
    {"U.S. Subclass": "122",        "Locarno Class - Subclass": "02 - 99", "Notes": ""},
    {"U.S. Subclass": "123",        "Locarno Class - Subclass": "02 - 06", "Notes": ""},
    {"U.S. Subclass": "124",        "Locarno Class - Subclass": "02 - 99", "Notes": ""},
    {"U.S. Subclass": "125 - 126",  "Locarno Class - Subclass": "29 - 01", "Notes": ""},
    {"U.S. Subclass": "127",        "Locarno Class - Subclass": "26 - 99", "Notes": ""},
    {"U.S. Subclass": "128",        "Locarno Class - Subclass": "29 - 02", "Notes": ""},
    {"U.S. Subclass": "129 - 130",  "Locarno Class - Subclass": "29 - 01", "Notes": ""},
]


In [37]:
D30 = [
    {"U.S. Subclass": "101 - 117",  "Locarno Class - Subclass": "30 - 02",
     "Notes": 'Classify "bird perch" as 30-07. Classify "tray for carrying pet" as 30-99.'},
    {"U.S. Subclass": "118",        "Locarno Class - Subclass": "30 - 06", "Notes": ""},
    {"U.S. Subclass": "119 - 120",  "Locarno Class - Subclass": "30 - 02", "Notes": ""},
    {"U.S. Subclass": "121 - 123",  "Locarno Class - Subclass": "30 - 07", "Notes": ""},
    {"U.S. Subclass": "124 - 133",  "Locarno Class - Subclass": "30 - 03", "Notes": ""},
    {"U.S. Subclass": "134 - 143",  "Locarno Class - Subclass": "30 - 04", "Notes": ""},
    {"U.S. Subclass": "144 - 150",  "Locarno Class - Subclass": "30 - 01", "Notes": 'Classify "animal shackle" as 30-08.'},
    {"U.S. Subclass": "151 - 153",  "Locarno Class - Subclass": "30 - 04", "Notes": ""},
    {"U.S. Subclass": "154",        "Locarno Class - Subclass": "30 - 09", "Notes": ""},
    {"U.S. Subclass": "155",        "Locarno Class - Subclass": "30 - 08", "Notes": ""},
    {"U.S. Subclass": "156 - 157",  "Locarno Class - Subclass": "30 - 05", "Notes": ""},
    {"U.S. Subclass": "158 - 159",  "Locarno Class - Subclass": "30 - 10", "Notes": ""},
    {"U.S. Subclass": "160",        "Locarno Class - Subclass": "30 - 12",
     "Notes": 'Classify "scratching posts for cats" as 30-06. Classify "artificial bones for dogs" as 30-12.'},
    {"U.S. Subclass": "161 - 162",  "Locarno Class - Subclass": "30 - 11", "Notes": ""},
    {"U.S. Subclass": "199",        "Locarno Class - Subclass": "30 - 99", "Notes": ""},
]

In [38]:
D32 = [
    {"U.S. Subclass": "1 - 18",    "Locarno Class - Subclass": "15 - 05", "Notes": ""},
    {"U.S. Subclass": "19",        "Locarno Class - Subclass": "04 - 03", "Notes": ""},
    {"U.S. Subclass": "20 - 34",   "Locarno Class - Subclass": "15 - 05", "Notes": ""},
    {"U.S. Subclass": "35 - 36",   "Locarno Class - Subclass": "07 - 05", "Notes": ""},
    {"U.S. Subclass": "37",        "Locarno Class - Subclass": "09 - 04", "Notes": ""},
    {"U.S. Subclass": "38 - 53",   "Locarno Class - Subclass": "07 - 05", "Notes": ""},
    {"U.S. Subclass": "41",        "Locarno Class - Subclass": "08 - 05", "Notes": ""},
    {"U.S. Subclass": "53.1",      "Locarno Class - Subclass": "09 - 02", "Notes": ""},
    {"U.S. Subclass": "53.1",      "Locarno Class - Subclass": "08 - 05", "Notes": ""},
    {"U.S. Subclass": "54 - 75",   "Locarno Class - Subclass": "07 - 05", "Notes": ""},
]

In [39]:
D34 = [
    {
        "U.S. Subclass": "1 - 1.1",
        "Locarno Class - Subclass": "23 - 99",
        "Notes": (
            'Classify "incinerator" as 23-99.\n'
            'Classify "napkin holders (sanitary equipment) as 23-08.\n'
            'Classify "holder for refuse bag" as 07-99.'
        ),
    },
    {"U.S. Subclass": "2 - 11",    "Locarno Class - Subclass": "09 - 09", "Notes": ""},
    {"U.S. Subclass": "12 - 27",   "Locarno Class - Subclass": "12 - 02", "Notes": ""},
    {"U.S. Subclass": "28 - 37",   "Locarno Class - Subclass": "12 - 05", "Notes": ""},
    {"U.S. Subclass": "38",        "Locarno Class - Subclass": "09 - 08", "Notes": ""},
    {"U.S. Subclass": "39",        "Locarno Class - Subclass": "09 - 02", "Notes": ""},
]

In [40]:
D99 = [
    {"U.S. Subclass": "1 - 4",     "Locarno Class - Subclass": "03 - 01", "Notes": ""},
    {"U.S. Subclass": "5",         "Locarno Class - Subclass": "06 - 13", "Notes": ""},
    {"U.S. Subclass": "6 - 9",     "Locarno Class - Subclass": "03 - 01", "Notes": ""},
    {"U.S. Subclass": "10",        "Locarno Class - Subclass": "06 - 04", "Notes": ""},
    {"U.S. Subclass": "11",        "Locarno Class - Subclass": "08 - 06", "Notes": ""},
    {"U.S. Subclass": "12",        "Locarno Class - Subclass": "03 - 01", "Notes": ""},
    {"U.S. Subclass": "13",        "Locarno Class - Subclass": "11 - 03", "Notes": ""},
    {"U.S. Subclass": "14",        "Locarno Class - Subclass": "08 - 08", "Notes": ""},
    {"U.S. Subclass": "15",        "Locarno Class - Subclass": "06 - 04", "Notes": ""},
    {"U.S. Subclass": "16",        "Locarno Class - Subclass": "11 - 05", "Notes": ""},
    {"U.S. Subclass": "17 - 24",   "Locarno Class - Subclass": "25 - 03", "Notes": 'Classify "tombstone" as 25-03.'},
    {"U.S. Subclass": "25 - 26",   "Locarno Class - Subclass": "11 - 01", "Notes": ""},
    {"U.S. Subclass": "27",        "Locarno Class - Subclass": "11 - 02", "Notes": ""},
    {
        "U.S. Subclass": "28",
        "Locarno Class - Subclass": "06 - 04",
        "Notes": (
            "Note the following exceptions for D99/28 29-43:\n"
            'Classify "burial vault" as 25-03.\n'
            'Classify "vault type safe" as 25-02.\n'
            'Classify "lock type or box safe" as 06-04.\n'
            'Classify "automatic teller machine" as 20-01.\n'
            'Classify "piggy banks" as 03-01.\n'
            'Classify "coin holders" as 03-01.\n'
            'Classify "cashboxes" as 19-02.'
        ),
    },
    {
        "U.S. Subclass": "28",
        "Locarno Class - Subclass": "25 - 02",
        "Notes": (
            "Note the following exceptions for D99/28 29-43:\n"
            'Classify "burial vault" as 25-03.\n'
            'Classify "vault type safe" as 25-02.\n'
            'Classify "lock type or box safe" as 06-04.\n'
            'Classify "automatic teller machine" as 20-01.\n'
            'Classify "piggy banks" as 03-01.\n'
            'Classify "coin holders" as 03-01.\n'
            'Classify "cashboxes" as 19-02.'
        ),
    },
    {"U.S. Subclass": "29 - 32",   "Locarno Class - Subclass": "03 - 01", "Notes": ""},
    {"U.S. Subclass": "33 - 43",   "Locarno Class - Subclass": "06 - 04", "Notes": ""},
]

In [41]:
CONVERSION_CHART = {
    "D1": D01, "D2": D02, "D3": D03, "D4": D04, "D5": D05,
    "D6": D06, "D7": D07, "D8": D08, "D9": D09, "D10": D10,
    "D11": D11, "D12": D12, "D13": D13, "D14": D14, "D15": D15,
    "D16": D16, "D17": D17, "D18": D18, "D19": D19, "D20": D20,
    "D21": D21, "D22": D22, "D23": D23, "D24": D24, "D25": D25,
    "D26": D26, "D27": D27, "D28": D28, "D29": D29, "D30": D30,
    "D32": D32, "D34": D34, "D99": D99
}

In [42]:
with open(main_dir / "USPC_CONVERSION_CHART.json", "w", encoding="utf-8") as f:
    json.dump(CONVERSION_CHART, f, ensure_ascii=False, indent=2)